# Analysis code for reco outputs
This save the cerenkov angles from gas and aerogel from a gaussian fit to a csv file


In [ ]:
import pandas as pd
%jsroot on
import ROOT
import re
all_file_results = []
directory_path="cuk-tests/1k/out"
output_csv_path = '1k_fit_parameters.csv'

if os.path.exists(directory_path) and os.path.isdir(directory_path):
    print(f"Processing ROOT files in directory: {directory_path}")
    for filename_in_dir in os.listdir(directory_path):
        if filename_in_dir.startswith('out_') and filename_in_dir.endswith('.root'):
            current_file_path = os.path.join(directory_path, filename_in_dir)
            print(f"\nExtracting histograms and fit parameters from file: {filename_in_dir}")
            
            file_results = {'filename': filename_in_dir}
            
            try:
                root_file = ROOT.TFile.Open(current_file_path)
                
                if not root_file or root_file.IsZombie():
                    print(f"  Error opening ROOT file: {filename_in_dir}")
                    all_file_results.append(file_results) # Add filename even if error
                    continue

                # Process 'ca' object and 'thtrA' histogram
                ca_object_root = root_file.Get("ca")
                if ca_object_root and 'TCanvas' in ca_object_root.ClassName():
                    primitives = ca_object_root.GetListOfPrimitives()
                    found_thtrA = False
                    for i in primitives:
                        # Check if 'i' is a TPad and matches the name where 'thtrA' is found
                        # Based on CELL INDEX 4, it's 'ca_8'
                        if 'TPad' in i.ClassName() and i.GetName() == "ca_8":
                            pad_primitives = i.GetListOfPrimitives()
                            for pp in pad_primitives:
                                if pp.GetName() == "thtrA":
                                    gaus_func = pp.GetFunction("gaus")
                                    if gaus_func:
                                        file_results['ca_thtrA_amplitude'] = gaus_func.GetParameter(0)
                                        file_results['ca_thtrA_amplitude_err'] = gaus_func.GetParError(0)
                                        file_results['ca_thtrA_mean'] = gaus_func.GetParameter(1)
                                        file_results['ca_thtrA_mean_err'] = gaus_func.GetParError(1)
                                        file_results['ca_thtrA_stddev'] = gaus_func.GetParameter(2)
                                        file_results['ca_thtrA_stddev_err'] = gaus_func.GetParError(2)
                                        file_results['ca_thtrA_chisquare'] = gaus_func.GetChisquare()
                                        file_results['ca_thtrA_ndf'] = gaus_func.GetNDF()
                                        file_results['ca_thtrA_entries'] = pp.GetEntries()
                                        found_thtrA = True
                                        print(f"  Extracted 'thtrA' fit parameters from 'ca'.")
                                    else:
                                        print(f"  'thtrA' histogram in 'ca' has no 'gaus' fit function.")
                                    break # Found thtrA, no need to check other pp
                            if found_thtrA:
                                break # Found the relevant TPad and thtrA, no need to check other i
                    if not found_thtrA:
                        print("  'thtrA' histogram not found within 'ca' object's pads or specific TPad 'ca_8'.")
                else:
                    print("  'ca' object not found or is not a TCanvas.")

                # Process 'cg' object and 'thtrG' histogram
                cg_object_root = root_file.Get("cg")
                if cg_object_root and 'TCanvas' in cg_object_root.ClassName():
                    primitives = cg_object_root.GetListOfPrimitives()
                    found_thtrG = False
                    for i in primitives:
                        # Iterate through all pads to find 'thtrG'
                        if 'TPad' in i.ClassName():
                            pad_primitives = i.GetListOfPrimitives()
                            for pp in pad_primitives:
                                if pp.GetName() == "thtrG":
                                    gaus_func = pp.GetFunction("gaus")
                                    if gaus_func:
                                        file_results['cg_thtrG_amplitude'] = gaus_func.GetParameter(0)
                                        file_results['cg_thtrG_amplitude_err'] = gaus_func.GetParError(0)
                                        file_results['cg_thtrG_mean'] = gaus_func.GetParameter(1)
                                        file_results['cg_thtrG_mean_err'] = gaus_func.GetParError(1)
                                        file_results['cg_thtrG_stddev'] = gaus_func.GetParameter(2)
                                        file_results['cg_thtrG_stddev_err'] = gaus_func.GetParError(2)
                                        file_results['cg_thtrG_chisquare'] = gaus_func.GetChisquare()
                                        file_results['cg_thtrG_ndf'] = gaus_func.GetNDF()
                                        file_results['cg_thtrG_entries'] = pp.GetEntries()
                                        found_thtrG = True
                                        print(f"  Extracted 'thtrG' fit parameters from 'cg'.")
                                    else:
                                        print(f"  'thtrG' histogram in 'cg' has no 'gaus' fit function.")
                                    break # Found thtrG, no need to check other pp
                            if found_thtrG:
                                break # Found TPad with thtrG, no need to check other i
                    if not found_thtrG:
                        print("  'thtrG' histogram not found within 'cg' object's pads.")
                else:
                    print("  'cg' object not found or is not a TCanvas.")
                
                root_file.Close()
                all_file_results.append(file_results)

            except Exception as e:
                print(f"  Error processing {filename_in_dir} with ROOT: {e}")
                all_file_results.append(file_results) # Append incomplete results to still get filename

    # Create DataFrame and save to CSV
    if all_file_results:
        df_results = pd.DataFrame(all_file_results)
        output_csv_path = 'fit_parameters.csv'
        

        # remove rows with missing fit parameters for mean Cerenkov angles (ca_thtrA_mean and cg_thtrG_mean)
        initial_row_count = len(df_results)
        # df_results = df_results.dropna(subset=['ca_thtrA_mean', 'cg_thtrG_mean']) This might create a problem: Deepak Samuel
        removed_row_count = initial_row_count - len(df_results)

        print(f"Removed {removed_row_count} incomplete rows from df_results (missing ca_thtrA_mean or cg_thtrG_mean).")
        print(f"DataFrame now contains {len(df_results)} rows.")


        # Extract PID, momentum, and pseudorapidity from the 'filename' column
        # The regex pattern looks for digits between underscores: out_PID_momentum_pseudorapidity_bin.root
        pattern = r'out_(\d+)_(\d+\.?\d*)_(\d+)\.root'
        extracted_data = df_results['filename'].str.extract(pattern)

        # Assign the extracted data to new columns, converting to appropriate types
        df_results['PID'] = extracted_data[0].astype(int)
        df_results['momentum'] = extracted_data[1].astype(float)
        df_results['pseudorapidity'] = extracted_data[2].astype(int)

        print("Added 'PID', 'momentum', and 'pseudorapidity' columns to df_results.")
        print(df_results[['filename', 'PID', 'momentum', 'pseudorapidity']].head())

        # Define the mapping from pseudorapidity bin index to actual pseudorapidity value
        # https://docs.google.com/spreadsheets/d/1qNh3giwhRENJi25z45lgeh5d7QwIw5Uy3lepISHCv9Y/edit?gid=2032570815#gid=2032570815
        pseudorapidity_map = {
            0: 1.5, 1: 1.6, 2: 1.7, 3: 1.8, 4: 1.9,
            5: 2.0, 6: 2.1, 7: 2.2, 8: 2.3, 9: 2.4,
            10: 2.5, 11: 2.6, 12: 2.7, 13: 2.8, 14: 2.9,
            15: 3.0, 16: 3.1, 17: 3.2, 18: 3.3, 19: 3.4,
            20: 3.5
        }

        # Create a new 'eta' column by mapping the 'pseudorapidity' (bin index) column
        df_results['eta'] = df_results['pseudorapidity'].map(pseudorapidity_map)

        df_results.to_csv(output_csv_path, index=False)
        print(f"\nAll fit parameters saved to {output_csv_path}")
    else:
        print("No ROOT files processed or no data extracted.")
else:
    print(f"Directory not found: {directory_path}")